# Action Recognition in Videos — UCF101 (Colab, v3)

Full 101-class pipeline: class-weighted loss, early stopping, bounded
hyperparameter search, and a step-by-step Streamlit frontend (run
locally after downloading results — see the last cell).

Repo: `Action-Recognition-in-Videos-ft-SRINATH-clzv3`

**Runtime:** Runtime → Change runtime type → GPU (T4 or better).

In [ ]:
# Clone and auto-detect the correct project folder, regardless of nesting.
# This fixes the "nested action-recognition folder" issue from before —
# it searches for the folder that actually contains src/ instead of
# assuming a fixed path.
import os

%cd /content
if not os.path.isdir("Action-Recognition-in-Videos-ft-SRINATH-clzv3"):
    !git clone https://github.com/SrinathRavi10/Action-Recognition-in-Videos-ft-SRINATH-clzv3.git
os.chdir("/content/Action-Recognition-in-Videos-ft-SRINATH-clzv3")

if not os.path.isdir("src"):
    found = False
    for root, dirs, files in os.walk("."):
        if "src" in dirs and os.path.isfile(os.path.join(root, "src", "config.py")):
            os.chdir(root)
            found = True
            break
    if not found:
        raise RuntimeError("Could not find a folder containing src/config.py — check the repo structure manually.")

print("Working directory:", os.getcwd())
print("Contents:", os.listdir("."))
assert os.path.isdir("src") and os.path.isfile("requirements.txt"), "Still not in the right folder!"
print("\nConfirmed: src/ and requirements.txt found. Ready to proceed.")

In [ ]:
# Install dependencies
!pip install -q -r requirements.txt

In [ ]:
# Confirm GPU is available
import torch
print("CUDA available:", torch.cuda.is_available())
print("Device:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")

## (Optional) Mount Google Drive to persist checkpoints across sessions

Recommended given full 101-class training takes a while — a Colab
disconnect shouldn't cost you the whole run.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
DRIVE_CHECKPOINT_DIR = "/content/drive/MyDrive/action-recognition-checkpoints"
os.makedirs(DRIVE_CHECKPOINT_DIR, exist_ok=True)

# Point the project's checkpoint directory at Drive instead of local (ephemeral) storage
import sys
sys.path.insert(0, ".")
import src.config as config
config.CHECKPOINT_DIR = DRIVE_CHECKPOINT_DIR
print("Checkpoints will be saved to:", config.CHECKPOINT_DIR)

## 1. Download the full 101-class dataset

This is a much larger download than a small subset — expect it to take a while.

In [ ]:
!python data/download_ucf101.py

## 2. (Optional) Bounded hyperparameter probe

Trains a few epochs across a small grid of learning rates/freeze depths, picks a starting configuration. Not an exhaustive search — see the README's Precautions table for why.

In [ ]:
!python src/hyperparam_search.py

In [ ]:
import json
with open("outputs/best_hparams.json") as f:
    hp = json.load(f)
print("Best probe config:", hp["best"])
print("\nIf you want to use this, edit LEARNING_RATE in src/config.py to match, then re-run the next cell.")

## 3. Train

Class-weighted loss, LR warmup + cosine decay, gradient clipping, early stopping. This will take noticeably longer than a small-subset run.

In [ ]:
!python src/train.py

## 4. Evaluate

In [ ]:
!python src/evaluate.py --checkpoint checkpoints/best_model.pt

In [ ]:
from IPython.display import Image, display
display(Image("outputs/confusion_matrix.png"))

## 5. Visualize predictions

In [ ]:
!python src/visualize_predictions.py --checkpoint checkpoints/best_model.pt --num_samples 8

from IPython.display import Image, display
display(Image("outputs/sample_predictions.png"))

## 6. Pose-overlay video visualizations

In [ ]:
!python src/visualize_pose_predictions.py --checkpoint checkpoints/best_model.pt --num_clips 5

## 7. Download everything you need for the frontend

The Streamlit frontend (`app/streamlit_app.py`) runs **locally on your
own machine**, not inline in Colab. Download these before your session
ends:
- `checkpoints/best_model.pt`
- the whole `outputs/` folder
- `data/UCF101_subset/classes.txt` and `data/UCF101_subset/class_distribution.json`

Then locally: `pip install -r requirements.txt` and
`streamlit run app/streamlit_app.py`.

In [ ]:
# Zip up everything the frontend needs, for a single one-click download
!zip -r frontend_artifacts.zip checkpoints/best_model.pt outputs data/UCF101_subset/classes.txt data/UCF101_subset/class_distribution.json

from google.colab import files
files.download("frontend_artifacts.zip")